# D128 — Mocking Basics

A **mock** is a controlled replacement for a dependency. It helps us test our code without sending a real email, calling an API, or connecting to a database.

Python provides mocks in the built-in `unittest.mock` module, so no installation is required.

## A small example

`OrderService` uses an email sender after placing an order. During a unit test, we do not want to send a real email. We only want to confirm that the service asks the sender to send the correct message.

In [ ]:
class EmailSender:
    def send(self, address, message):
        # A real implementation might contact an email server.
        raise NotImplementedError


class OrderService:
    def __init__(self, email_sender):
        self.email_sender = email_sender

    def place_order(self, customer_email):
        message = "Your order was placed"
        self.email_sender.send(customer_email, message)
        return "confirmed"

## 1. Create and use a mock

`Mock(spec=EmailSender)` creates a mock that follows the public interface of `EmailSender`. Using a `spec` is safer than an unrestricted mock because misspelled or unknown attributes raise an error.

In [ ]:
from unittest.mock import Mock

email_sender = Mock(spec=EmailSender)
service = OrderService(email_sender)

result = service.place_order("learner@example.com")

assert result == "confirmed"

## 2. Verify how the mock was called

A mock records its calls. `assert_called_once_with` checks that the method was called exactly once with the expected arguments.

In [ ]:
email_sender.send.assert_called_once_with(
    "learner@example.com",
    "Your order was placed",
)

print("The expected email request was made")

## 3. Control a return value

Use `return_value` when the dependency must return a predictable result.

In [ ]:
price_reader = Mock()
price_reader.get_price.return_value = 500

price = price_reader.get_price("Mouse")

assert price == 500
price_reader.get_price.assert_called_once_with("Mouse")

## 4. Simulate an error with `side_effect`

A `side_effect` can raise an exception. This is useful for checking how code behaves when a dependency fails.

In [ ]:
failing_sender = Mock(spec=EmailSender)
failing_sender.send.side_effect = ConnectionError("Email service unavailable")

try:
    failing_sender.send("learner@example.com", "Hello")
except ConnectionError as error:
    assert str(error) == "Email service unavailable"
    print("The dependency failure was simulated")

## Using the example with pytest

The basic test can be written in `tests/test_order_service.py`:

```python
from unittest.mock import Mock

from shop.orders import EmailSender, OrderService


def test_place_order_requests_confirmation_email():
    email_sender = Mock(spec=EmailSender)
    service = OrderService(email_sender)

    result = service.place_order("learner@example.com")

    assert result == "confirmed"
    email_sender.send.assert_called_once_with(
        "learner@example.com",
        "Your order was placed",
    )
```

Run it from the project root with `python -m pytest`.

## Keep mocks reliable

- Mock an external dependency, not the simple code being tested.
- Prefer `Mock(spec=RealType)` when a real interface is available.
- Verify only important interactions. Too many call checks make tests fragile.
- Keep the mock close to the test so its behavior is easy to see.

For now, remember the four basic tools: `Mock`, `spec`, `return_value`, and `side_effect`. Later notebooks can cover patching and more advanced mock behavior.